In [1]:
from get.get import *
from calculate.calculate import *
from visualize.visualize import *
from db.db import *

2026-04-02 17:03:22.240 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


In [2]:
# Get data building data
df_grid, grid_bd_points = get_latest_grid_data()
dfs1 = get_dfs1()

# Get pop/density data
dfs2 = get_dfs2(df_grid)

# Set score
weight_dic = {'broadcast': 0.41787,
             'electricity':0.02198,
             'factory':0.00376,
             'hospital':0.00064,
             'infra':0.00075,
             'prison':0.02757,
             'public':0.01237,
             'science':0.08148,
             'telecommunication':0.14754,
             'transportation':0.00187,
             'water':0.17237,
             'frequency':0}
set_score(dfs1, weight_dic)

# Parameters
RANGE_KM = 2.0

# Rank calculation
rank_dic, max_radar_num = calc_rank(dfs1, df_grid, RANGE_KM, radar_num=50, polygon_coords=grid_bd_points)

df_population = dfs2['population']
df_area_density = dfs2['area_density']

df_final = get_df_final(rank_dic, df_grid, df_population, df_area_density, RANGE_KM)

upload_result(df_final)

2026-04-01 14:06:44.505 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-04-01 14:06:44.507 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


1/2단계: DB 연결 및 체크 성공


d:\workspaces\00_Project\crawlProject\test_project\project\get_data\get.py:103: UserWarning: Geometry column does not contain geometry.
  gdf['geometry'] = gdf['geometry'].apply(lambda x: str(x))
2026-04-01 14:06:46.995 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


------------------------------
1순위
위치 : [ 37.528251 126.965614]
점수 : 0.00563
남은 시설물 : 0개
------------------------------
최대 radar 개수: 1
저장 완료: case6


In [3]:
# Icon config
ICON_MAP = {
    "broadcast":         folium.Icon(color="orange",     icon="broadcast-tower",   prefix="fa"),
    "electricity":       folium.Icon(color="green",      icon="bolt",              prefix="fa"),
    "factory":           folium.Icon(color="blue",       icon="industry",          prefix="fa"),
    "hospital":          folium.Icon(color="red",        icon="hospital",          prefix="fa"),
    "infra":             folium.Icon(color="darkblue",   icon="cogs",              prefix="fa"),
    "prison":            folium.Icon(color="black",      icon="university",        prefix="fa"),
    "public":            folium.Icon(color="cadetblue",  icon="building",          prefix="fa"),
    "science":           folium.Icon(color="pink",       icon="flask",             prefix="fa"),
    "telecommunication": folium.Icon(color="beige",      icon="satellite-dish",    prefix="fa"),
    "transportation":    folium.Icon(color="darkgreen",  icon="train",             prefix="fa"),
    "water":             folium.Icon(color="lightblue",  icon="tint",              prefix="fa"),
    "frequency":         folium.Icon(color="darkred",    icon="signal",            prefix="fa"),
}
visualize(df_grid, dfs1, rank_dic, RANGE_KM, ICON_MAP,
              show_rank=None, polygon_coords=grid_bd_points,
              df_final=df_final)

저장 완료: map.html


In [9]:
delete_result('case2')

In [2]:
# Get data building data
df_grid, grid_bd_points = get_latest_grid_data()
dfs1 = get_dfs1_server()

# Get pop/density data
dfs2 = get_dfs2_server(df_grid)

# Set score
weight_dic = {'broadcast': 0.41787,
             'electricity':0.02198,
             'factory':0.00376,
             'hospital':0.00064,
             'infra':0.00075,
             'prison':0.02757,
             'public':0.01237,
             'science':0.08148,
             'telecommunication':0.14754,
             'transportation':0.00187,
             'water':0.17237,
             'frequency':0}
set_score(dfs1, weight_dic)

# Parameters
RANGE_KM = 2.0

# Rank calculation
rank_dic, max_radar_num = calc_rank(dfs1, df_grid, RANGE_KM, radar_num=50, polygon_coords=grid_bd_points)

df_population = dfs2['population']
df_area_density = dfs2['area_density']

df_final = get_df_final(rank_dic, df_grid, df_population, df_area_density, RANGE_KM)


d:\workspaces\00_Project\crawlProject\test_project\project\get_data\get.py:64: SAWarning: Unrecognized server version info '17.0.1000.7'.  Some SQL Server features may not function properly.
  with temp_engine.connect() as conn:
2026-04-01 16:42:50.131 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


🔄 기존 데이터베이스(projectdb) 삭제 중...
🆕 새 데이터베이스(projectdb) 생성 중...
✅ 'projectdb' 초기화 및 생성 완료!


d:\workspaces\00_Project\crawlProject\test_project\project\get_data\get.py:132: SAWarning: Unrecognized server version info '17.0.1000.7'.  Some SQL Server features may not function properly.
  with engine.connect() as conn:


1/2단계: DB 연결 및 체크 성공


d:\workspaces\00_Project\crawlProject\test_project\project\get_data\get.py:169: UserWarning: Geometry column does not contain geometry.
  gdf['geometry'] = gdf['geometry'].apply(lambda x: str(x))


------------------------------
1순위
위치 : [ 37.528251 126.965614]
점수 : 0.00563
남은 시설물 : 0개
------------------------------
최대 radar 개수: 1


In [6]:
engine = get_engine()
db = st.secrets['mysql']


In [8]:
from sqlalchemy import inspect
inspector = inspect(engine)
existing_tables = inspector.get_table_names()
print(existing_tables)

['broadcast', 'density', 'electricity', 'factory', 'frequency', 'hospital', 'infra', 'population_raw', 'prison', 'public', 'science', 'telecommunication', 'transportation', 'water']


In [22]:
file_list        = glob('final_data/*.csv')
exclude_keywords = ['population_raw', 'density']  # 중간 처리용 테이블 제외
table_names      = []

for file in file_list:
    file_basename = os.path.basename(file)
    raw_name   = os.path.splitext(file_basename)[0]
    table_name = raw_name[3:] if raw_name.startswith('df_') else raw_name
    table_names.append(table_name)

json_list = glob('final_data/*.geojson')
for json_file in json_list:
    raw_name  = os.path.splitext(os.path.basename(json_file))[0]
    json_name = raw_name.split('_')[-1]
    table_names.append(json_name)

table_names

['broadcast',
 'electricity',
 'factory',
 'frequency',
 'hospital',
 'infra',
 'population_raw',
 'prison',
 'public',
 'science',
 'telecommunication',
 'transportation',
 'water',
 'density']

In [ ]:
visualize()

In [ ]:
visualize(df_grid, dfs, rank_dic, RANGE_KM, ICON_MAP,
              show_rank=None, polygon_coords=None,
              df_final=None)

In [3]:
dfs1 = get_dfs1()

In [5]:
dfs1.keys()

dict_keys(['broadcast', 'electricity', 'factory', 'frequency', 'hospital', 'infra', 'prison', 'public', 'science', 'telecommunication', 'transportation', 'water'])

In [11]:
df_grid, bd = get_latest_grid_data()

경고: 짝이 되는 JSON(grid (2)_polygon.json)이 없다.
 데이터 로드 중 오류 발생: 'NoneType' object is not subscriptable


In [8]:
dfs2 = get_dfs2(df_grid)